## Hello Data!
Load raw CSV, display first 3 rows

In [ ]:
import pandas as pd
import plotly as plt

data = pd.read_csv("data/5000 Sales Records.csv")
data.head(3)

,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Central America and the Caribbean,Antigua and Barbuda,Baby Food,Online,M,12/20/2013,957081544,1/11/2014,552,255.28,159.42,140914.56,87999.84,52914.72
1,Central America and the Caribbean,Panama,Snacks,Offline,C,7/5/2010,301644504,7/26/2010,2167,152.58,97.44,330640.86,211152.48,119488.38
2,Europe,Czech Republic,Beverages,Offline,C,9/12/2011,478051030,9/29/2011,4778,47.45,31.79,226716.10,151892.62,74823.48


## Pick the Right Container
Justify dict vs namedtuple vs sets(1–2 sentences)

Sets cannot have identical values, which is a possibility in the dataset, rejected.
Dicts and namedtuples are similar in this usecase, but dicts are mutable and can add more "fields". We will use dicts just in case mutability is needed later.

## 	Implement Functions and  Data structure
Implement and use it to populate a data structure

In [2]:
from src.structure import Transaction


## Bulk Loaded
Example: Map data structures from dataframes to dictionaries

In [8]:
dicts = data.to_dict(orient="records")
transactions = [Transaction(d) for d in dicts]

## Quick Profiling
Min/mean/max price, unique city count (set)
**NOTE: The suggested dataset in the assignment instructions does NOT have these fields. However the purpose of this lab is simply to demonstrate skills - not find unique city count. I have found min/mean/max unit price, and unique countries instead.**


In [5]:
min = min(t.unit_price for t in transactions)
mean = sum(t.unit_price for t in transactions) / len(transactions)
max = max(t.unit_price for t in transactions)

uniquecountries = set(t.country for t in transactions)

print (min, mean, max)
print(uniquecountries)

9.33 265.745564 668.27
{'Bhutan', 'Cameroon', 'Swaziland', 'Cuba', 'Bahrain', 'Mexico', 'Mauritius ', 'Kyrgyzstan', 'United Arab Emirates', 'Israel', 'Sudan', 'Malta', 'East Timor', 'The Bahamas', 'Yemen', 'Kiribati', 'Rwanda', 'Maldives', 'Lesotho', 'Estonia', 'Slovenia', 'Belize', 'Georgia', 'Greece', 'Costa Rica', 'Andorra', 'Benin', 'El Salvador', 'Cape Verde', 'Thailand', 'Kazakhstan', 'Japan', 'Grenada', 'Tonga', 'Federated States of Micronesia', 'Switzerland', 'Romania', 'Syria', 'The Gambia', 'Kosovo', 'Nauru', 'Nigeria', 'Burkina Faso', 'France', 'Equatorial Guinea', 'Russia', 'Ukraine', 'Panama', 'Honduras', 'Namibia', 'South Sudan', 'Guinea', 'Mongolia', 'Finland', 'Nepal', 'Egypt', 'Vietnam', 'Sao Tome and Principe', 'Oman', 'Bosnia and Herzegovina', 'Mauritania', 'Luxembourg', 'Montenegro', 'Norway', 'Angola', 'Libya', 'Lithuania', 'Togo', 'Macedonia', 'United Kingdom', 'Niger', 'Lebanon', 'South Africa', 'Guinea-Bissau', 'Tuvalu', 'Bangladesh', 'Bulgaria', 'Spain', 'Sierr

## Spot the Grime
Identify at least three dirty data cases

The Country field has several instances of leading/trailing whitespace which is inconsistent with the rest of the data.
Examples are 'Antigua and Barbuda ', 'Mauritius ', 'Samoa ', and 'Seychelles '.

## Cleaning Rules
Execute fixes inside clean(); show “before/after” counts

In [ ]:
dirtybefore = [t for t in transactions
    if any(
        isinstance(getattr(t, field), str) and getattr(t, field).strip() != getattr(t, field)
        for field in vars(t)
    )
]
print(len(dirtybefore))

for t in transactions:
    t.clean()
    
dirtyafter = [t for t in transactions
    if any(
        isinstance(getattr(t, field), str) and getattr(t, field).strip() != getattr(t, field)
        for field in vars(t)
    )
]
print(len(dirtyafter))

216
0


## Transformations
For example: Parse coupon_code ➞ numeric discount (others apply)

The field priority has Char values of L,C,H, and M. Not sure what these mean but lets convert them to 0,1,2,3 respectively.
Additionally, we can convert all str dates to datetime objects.

In [10]:
from datetime import datetime

for t in transactions:
    match t.order_priority:
        case "L":
            t.order_priority = 0
        case "C":
            t.order_priority = 1
        case "H":
            t.order_priority = 2
        case "M":
            t.order_priority = 3
        case _:
            pass

    t.order_date = datetime.strptime(t.order_date, "%m/%d/%Y")
    t.ship_date = datetime.strptime(t.ship_date, "%m/%d/%Y")

print(transactions[0].order_priority)
print(type(transactions[0].order_date))
print(type(transactions[0].ship_date))

3
<class 'datetime.datetime'>
<class 'datetime.datetime'>


## Feature Engineering
For example: Add days_since_purchase

In [11]:
for t in transactions:
    t.unit_profit = t.unit_price - t.unit_cost
    t.shipping_time = t.ship_date - t.order_date

print(transactions[0].unit_profit)
print(transactions[0].shipping_time)

95.86000000000001
22 days, 0:00:00


## Mini-Aggregation
For example: Revenue per shipping_city (dict or pandas.groupby)

In [15]:
#flag
country_totals = (
    data.groupby("Country", as_index=False)[["Units Sold", "Total Profit"]]
    .sum()
)

print(country_totals.head(3))

       Country  Units Sold  Total Profit
0  Afghanistan       77737    6010731.93
1      Albania       90608   10490706.02
2      Algeria      133198    7453132.79


## Serialization Checkpoint
Save cleaned data to JSON

In [ ]:
import json

with open("data/cleaned_transactions.json", "w") as output_file:
    json.dump([vars(t) for t in transactions], output_file, default=str, indent=2)


Saved 5000 transactions


## Soft Interview Reflection
Markdown: < 120 words explaining how Functions have helped

Functions have been extremely helpful in keeping code logical, simple, and organized. They can abstract a complicated code process, and for the future can make it easier to understand and perform repetitive or extensive actions. Functions imported from other libraries were immensely helpful in providing already-programmed functionality to help with some of the tasks.

## Data-Dictionary Section
Merge field definitions from the primary CSV header and the secondary metadata source. 
Present as a tidy Markdown table including the new columns, for example: Field, Type, Description, Source. Explain how they were created, e.g. synthetic, combination, mean, etc.

https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes/blob/master/all/all.csv
The secondary dataset was sourced from the given link, which contains the regional codes for all countries. We can use this data to get numerical transformations of str based country names.



| Field | Type | Description | Source | Creation Method |
| :--- | :--- | :--- | :--- | :--- |
| `Region` | String | Geographic region name | Primary CSV | Original, cleaned |
| `Country` | String | Name of destination country | Primary CSV | Original, cleaned |
| `Item Type` | String | Product classification category | Primary CSV | Original |
| `Sales Channel` | String | Point of sale channel (e.g., Online, Offline) | Primary CSV | Original |
| `Order Priority` | String | Priority code (e.g., L, M, H, C) | Primary CSV | Original |
| `Order Date` | DateTime | Date when purchase order was placed | Primary CSV | Converted string format to ISO-8601 DateTime |
| `Order ID` | Integer | Unique identification number for transaction | Primary CSV | Original |
| `Ship Date` | DateTime | Date when items were shipped | Primary CSV | Converted string format to ISO-8601 DateTime |
| `Units Sold` | Integer | Number of product items ordered | Primary CSV | Original |
| `Unit Price` | Float64 | Retail selling price per individual unit ($) | Primary CSV | Original |
| `Unit Cost` | Float64 | Wholesale production/handling cost per unit ($) | Primary CSV | Original |
| `Total Revenue` | Float64 | Gross revenue generated prior to costs ($) | Primary CSV | Original |
| `Total Cost` | Float64 | Direct cumulative cost for items sold ($) | Primary CSV | Original |
| `Total Profit` | Float64 | Net profit generated from order ($) | Primary CSV | Original |
| `alpha_2` | String | 2-letter standardized ISO country code | Secondary CSV | Merged from ISO metadata via `Country` |
| `alpha_3` | String | 3-letter standardized ISO country code | Secondary CSV | Merged from ISO metadata via `Country` |
| `country_code` | Integer | Numerical ISO-3166 country identifier | Secondary CSV | Merged from ISO metadata via `Country` |
| `sub_region` | String | Refined geographic sub-region | Secondary CSV | Merged from ISO metadata via `Country` |
| `shipping_time` | Integer | Elapsed processing time between order and shipment | Derived | Difference: `(Ship Date - Order Date).dt.days` |
| `Unit_Profit` | Float64 | Net profit per unit | Derived | `(Unit Price - Unit Cost)` |
| `order_year` | Integer | Calendar year order was placed | Derived | Extracted from `Order Date` component |